# 01 - Data cleaning

First pass over the raw Telco file: what the columns look like, which ones need type fixes,
whether there are duplicates or outliers worth worrying about, and writing out a clean copy that
the rest of the notebooks (and the app) build on.

The actual cleaning logic lives in `src/data.py` so the Streamlit app applies exactly the same
transformations at scoring time.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import CATEGORICAL_COLUMNS, CLEAN_PATH, NUMERIC_COLUMNS, RAW_PATH, clean, load_raw, missing_total_charges

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [2]:
raw = load_raw(RAW_PATH)
print(raw.shape)
raw.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Column types

`SeniorCitizen` comes in as 0/1 while every other flag is Yes/No, and `TotalCharges` is read as
`object` even though it is a dollar amount. Everything else is what I expect.

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
raw.isna().sum().sort_values(ascending=False).head()

customerID          0
DeviceProtection    0
TotalCharges        0
MonthlyCharges      0
PaymentMethod       0
dtype: int64

`isna()` reports nothing, but that is misleading: the blanks in `TotalCharges` are a single
space character, which pandas reads as a perfectly valid string. Checking the column after
stripping whitespace tells a different story.

In [5]:
missing = missing_total_charges(raw)
print(len(missing), "rows with a blank TotalCharges")
missing[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11 rows with a blank TotalCharges


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


All eleven are customers with `tenure == 0` - they signed up this month and have not been
billed yet, so there is genuinely nothing to total. None of them churned. Rather than drop them
or let them silently become zero somewhere downstream, `clean()` sets `TotalCharges` to 0 only
for this first-month case and would leave any other blank as NaN, which the check below would
catch.

## Categorical values

Checking every categorical column for stray spellings or unexpected levels.

In [6]:
for col in CATEGORICAL_COLUMNS:
    print(f"{col:18s} {raw[col].unique().tolist()}")

gender             ['Female', 'Male']
SeniorCitizen      [0, 1]
Partner            ['Yes', 'No']
Dependents         ['No', 'Yes']
PhoneService       ['No', 'Yes']
MultipleLines      ['No phone service', 'No', 'Yes']
InternetService    ['DSL', 'Fiber optic', 'No']
OnlineSecurity     ['No', 'Yes', 'No internet service']
OnlineBackup       ['Yes', 'No', 'No internet service']
DeviceProtection   ['No', 'Yes', 'No internet service']
TechSupport        ['No', 'Yes', 'No internet service']
StreamingTV        ['No', 'Yes', 'No internet service']
StreamingMovies    ['No', 'Yes', 'No internet service']
Contract           ['Month-to-month', 'One year', 'Two year']
PaperlessBilling   ['Yes', 'No']
PaymentMethod      ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']


The service columns use `No internet service` / `No phone service` as a third level. That is
redundant with `InternetService == 'No'` / `PhoneService == 'No'`, so the cleaning step folds it
into `No` - it keeps the one-hot encoding smaller and avoids perfectly collinear dummies.

## Duplicates

`customerID` should be unique; checking both the id and full-row duplicates.

In [7]:
print("duplicate ids :", raw["customerID"].duplicated().sum())
print("duplicate rows:", raw.drop(columns="customerID").duplicated().sum())

duplicate ids : 0
duplicate rows: 22


No duplicate customers - there are a few rows that are identical apart from the id, which is
plausible for a dataset this size (customers on the same plan with the same tenure), so I keep
them.

## Apply the cleaning

`clean()` parses `TotalCharges` to float (zero for the first-month customers above), maps
`SeniorCitizen` to Yes/No, collapses the redundant service levels and encodes `Churn` as 0/1.

In [8]:
df = clean(raw)
assert df["TotalCharges"].notna().all(), "unexpected blank TotalCharges beyond the first-month customers"
df.dtypes

customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

In [9]:
df[NUMERIC_COLUMNS].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


## Outliers

IQR fences on the three numeric columns. `TotalCharges` is right-skewed simply because it is
tenure x monthly charge, so a long tail is expected there rather than an error.

In [10]:
def iqr_outliers(s: pd.Series) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

pd.Series({c: iqr_outliers(df[c]) for c in NUMERIC_COLUMNS}, name="iqr_outliers")

tenure            0
MonthlyCharges    0
TotalCharges      0
Name: iqr_outliers, dtype: int64

Zero points fall outside the IQR fences, so nothing is clipped or removed. The charge columns
are bounded by the plan catalogue anyway (there is no such thing as a $10,000 monthly plan).

## Target balance

In [11]:
df["Churn"].value_counts(normalize=True).rename("share").round(3)

Churn
0    0.735
1    0.265
Name: share, dtype: float64

About 26.5% of customers churned. That is imbalanced enough that plain accuracy will be a
misleading headline number (predicting "stays" for everyone already scores 73%), which is why the
modelling notebook resamples with SMOTE and reports recall, F1 and ROC-AUC.

In [12]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("saved", CLEAN_PATH.relative_to(PROJECT_ROOT), df.shape)

saved data/processed/telco_clean.csv (7043, 21)
